# Natural-Language Mobility Query Interface

This notebook overlays a safe natural-language interface on top of the cleaned mobility dataset.

It supports:
- top route analysis
- inter-cell handover analysis
- intent parsing and safe SQL generation
- optional route mapping if a cell coordinate table is available

Expected data sources:
- `/home/jovyan/data/stage/handover_events/**/*.parquet`
- `/home/jovyan/data/stage/trips/**/*.parquet`


In [ ]:
# Section 1: Load Clean Mobility Dataset and Define Queryable Tables

from __future__ import annotations

import re
import json
import math
from dataclasses import dataclass, asdict
from datetime import date, timedelta
from pathlib import Path

import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# pyright: ignore[reportMissingImports]  # Imports are available in active kernel
HANDOVER_GLOB = "/home/jovyan/data/stage/handover_events/**/*.parquet"
TRIPS_GLOB = "/home/jovyan/data/stage/trips/**/*.parquet"
ANALYTICS_DB = "/home/jovyan/data/analytics.duckdb"

if not Path("/home/jovyan/data/stage/handover_events").exists():
    raise FileNotFoundError("Missing handover stage dir: /home/jovyan/data/stage/handover_events")
if not Path("/home/jovyan/data/stage/trips").exists():
    raise FileNotFoundError("Missing trips stage dir: /home/jovyan/data/stage/trips")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE VIEW handover_events AS SELECT * FROM read_parquet('{HANDOVER_GLOB}')")
con.execute(f"CREATE OR REPLACE VIEW trips AS SELECT * FROM read_parquet('{TRIPS_GLOB}')")

# Enforce explicit types in normalized views used by the NL layer.
con.execute("""
CREATE OR REPLACE VIEW handover_events_typed AS
SELECT
    CAST(vehicle_id AS VARCHAR) AS vehicle_id,
    CAST(imsi AS VARCHAR) AS imsi,
    CAST(event_ts AS TIMESTAMP) AS event_ts,
    CAST(cell_id AS BIGINT) AS cell_id,
    CAST(rat AS VARCHAR) AS rat,
    CAST(source_file AS VARCHAR) AS source_file,
    CAST(event_date AS DATE) AS event_date
FROM handover_events
""")

con.execute("""
CREATE OR REPLACE VIEW trips_typed AS
SELECT
    CAST(vehicle_id AS VARCHAR) AS vehicle_id,
    CAST(imsi AS VARCHAR) AS imsi,

SyntaxError: incomplete input (2874794661.py, line 45)

## Section 1b: Build Cell Locations from IP Mapping

Joins raw event files (column 8 = ip_address, column 18 = cell_id) with
`shared_ip_mapping.csv` to produce a `cell_locations` table in analytics.duckdb.

Join chain: `raw_event[col8=ip_address, col18=cell_id]` → `shared_ip_mapping[ip_address → location(lat:lon)]`

One-time build — skipped if `cell_locations` already exists in analytics.duckdb.


In [ ]:
import time

IP_MAPPING_CSV = "/home/jovyan/data/sim/raw/shared_ip_mapping.csv"
RAW_EVENTS_DIR = Path("/home/jovyan/data/sim/raw/sim_test_decoded")
ANALYTICS_DB   = "/home/jovyan/data/analytics.duckdb"

# How many rows to sample from each gz file.
# 639 files × 3000 rows × 2 cols ≈ trivial; covers virtually all unique cell_ids.
ROWS_PER_FILE = 3_000


def _cell_locations_exists() -> bool:
    a = duckdb.connect(ANALYTICS_DB, read_only=True)
    try:
        return "cell_locations" in {r[0] for r in a.execute("SHOW TABLES").fetchall()}
    finally:
        a.close()


if _cell_locations_exists():
    print("✓ cell_locations already present in analytics.duckdb — skipping build.")
else:
    print("Building cell_locations (sampled head of each file) …")
    t0 = time.time()

    # ── Step 1: IP → lat/lon dict (utf-8-sig strips the BOM) ─────────────────
    ip_df = pd.read_csv(IP_MAPPING_CSV, encoding="utf-8-sig", usecols=["ip_address", "location"], dtype=str)
    ip_df["ip_address"] = ip_df["ip_address"].str.strip()
    loc_parts = ip_df["location"].str.split(":", expand=True)
    ip_df["lat"] = pd.to_numeric(loc_parts[0], errors="coerce")
    ip_df["lon"] = pd.to_numeric(loc_parts[1], errors="coerce")
    ip_df = ip_df.dropna(subset=["lat", "lon"])
    ip_coords: dict[str, tuple[float, float]] = dict(
        zip(ip_df["ip_address"], zip(ip_df["lat"], ip_df["lon"]))
    )
    print(f"  IP mapping: {len(ip_coords):,} entries  ({time.time()-t0:.1f}s)")

    # ── Step 2: collect unique cell_ids we need to resolve ────────────────────
    target_cells: set[int] = set(
        con.execute(
            "SELECT DISTINCT cell_id FROM handover_events_typed WHERE cell_id IS NOT NULL"
        ).df()["cell_id"].tolist()
    )
    print(f"  Unique cell_ids in dataset: {len(target_cells):,}")

    # ── Step 3: sample head of each gz file (2 cols, ROWS_PER_FILE rows) ──────
    gz_files = sorted(RAW_EVENTS_DIR.rglob("*.gz"))
    print(f"  Sampling first {ROWS_PER_FILE:,} rows × {len(gz_files):,} files …")

    cell_to_ip: dict[int, str] = {}
    errors = 0

    for i, gz_path in enumerate(gz_files, 1):
        if not (target_cells - set(cell_to_ip.keys())):
            print(f"  Early exit: all cells resolved at file {i - 1}.")
            break
        try:
            chunk = pd.read_csv(
                gz_path,
                header=None,
                usecols=[8, 18],
                dtype=str,
                compression="gzip",
                nrows=ROWS_PER_FILE,
                on_bad_lines="skip",
                engine="c",
            )
            chunk.columns = ["ip", "cell_raw"]
            chunk["cell_id"] = pd.to_numeric(chunk["cell_raw"], errors="coerce")
            chunk = chunk.dropna(subset=["cell_id"])
            chunk["cell_id"] = chunk["cell_id"].astype(int)
            # only rows for cells we still need that also have a known IP
            needed = target_cells - set(cell_to_ip.keys())
            chunk = chunk[chunk["cell_id"].isin(needed) & chunk["ip"].isin(ip_coords)]
            for cell_id, ip in chunk.drop_duplicates("cell_id").set_index("cell_id")["ip"].items():
                cell_to_ip[cell_id] = ip
        except Exception as exc:
            errors += 1

    resolved = len(cell_to_ip)
    missed   = len(target_cells) - resolved
    print(f"  Resolved: {resolved:,}  |  Missed: {missed:,}  |  Errors: {errors}  ({time.time()-t0:.1f}s)")

    # ── Step 3b: second pass — full read for any still-missing cells ──────────
    if missed > 0:
        remaining = target_cells - set(cell_to_ip.keys())
        print(f"  Second pass for {len(remaining):,} unresolved cells (full file read) …")
        for gz_path in gz_files:
            if not remaining:
                break
            try:
                chunk = pd.read_csv(
                    gz_path, header=None, usecols=[8, 18], dtype=str,
                    compression="gzip", on_bad_lines="skip", engine="c",
                )
                chunk.columns = ["ip", "cell_raw"]
                chunk["cell_id"] = pd.to_numeric(chunk["cell_raw"], errors="coerce")
                chunk = chunk.dropna(subset=["cell_id"])
                chunk["cell_id"] = chunk["cell_id"].astype(int)
                chunk = chunk[chunk["cell_id"].isin(remaining) & chunk["ip"].isin(ip_coords)]
                for cell_id, ip in chunk.drop_duplicates("cell_id").set_index("cell_id")["ip"].items():
                    cell_to_ip[cell_id] = ip
                    remaining.discard(cell_id)
            except Exception:
                pass
        print(f"  After second pass: {len(cell_to_ip):,} resolved, {len(remaining):,} still unresolved")

    # ── Step 4: build DataFrame and write to analytics.duckdb ────────────────
    loc_df = pd.DataFrame([
        {"cell_id": cid, "lat": ip_coords[ip][0], "lon": ip_coords[ip][1]}
        for cid, ip in cell_to_ip.items()
    ]).drop_duplicates("cell_id").sort_values("cell_id")

    a = duckdb.connect(ANALYTICS_DB)
    try:
        a.execute("DROP TABLE IF EXISTS cell_locations")
        a.execute("CREATE TABLE cell_locations AS SELECT * FROM loc_df")
        n = a.execute("SELECT COUNT(*) FROM cell_locations").fetchone()[0]
        print(f"  ✓ Written to analytics.duckdb: {n:,} cell_locations  (total: {time.time()-t0:.1f}s)")
    finally:
        a.close()

# ── Register as a view on the session con ────────────────────────────────────
a = duckdb.connect(ANALYTICS_DB, read_only=True)
try:
    loc_df_loaded = a.execute("SELECT * FROM cell_locations").df()
finally:
    a.close()

con.execute("CREATE OR REPLACE VIEW cell_locations AS SELECT * FROM loc_df_loaded")

total = con.execute("SELECT COUNT(*) FROM cell_locations").fetchone()[0]
print(f"\ncell_locations in session: {total:,} cells")
con.execute("SELECT * FROM cell_locations ORDER BY cell_id LIMIT 5").df()


Building cell_locations (sampled head of each file) …
  IP mapping: 7,947 entries  (0.0s)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  Unique cell_ids in dataset: 1,042,854
  Sampling first 3,000 rows × 639 files …
  Resolved: 205,571  |  Missed: 837,283  |  Errors: 142  (251.5s)
  Second pass for 837,283 unresolved cells (full file read) …


KeyboardInterrupt: 

## Section 2: Create Aggregation Views for Routes and Inter-Cell Handovers

Derived views provide reusable route popularity and inter-cell transition metrics.

In [ ]:
con.execute("""
CREATE OR REPLACE VIEW route_popularity AS
SELECT
    first_cell_id AS source_cell,
    last_cell_id AS target_cell,
    COUNT(*) AS route_count,
    COUNT(DISTINCT vehicle_id) AS unique_vehicles,
    AVG(duration_minutes) AS avg_duration_min,
    AVG(n_handovers) AS avg_handovers,
    MIN(event_date) AS first_seen,
    MAX(event_date) AS last_seen
FROM trips_with_cells
WHERE first_cell_id IS NOT NULL
  AND last_cell_id IS NOT NULL
  AND first_cell_id <> last_cell_id
GROUP BY 1, 2
""")

con.execute("""
CREATE OR REPLACE VIEW intercell_handover_counts AS
WITH ranked AS (
    SELECT
        vehicle_id,
        event_ts,
        event_date,
        cell_id AS target_cell,
        LAG(cell_id) OVER (PARTITION BY vehicle_id ORDER BY event_ts) AS source_cell
    FROM handover_events_typed
), clean AS (
    SELECT
        source_cell,
        target_cell,
        event_date
    FROM ranked
    WHERE source_cell IS NOT NULL
      AND target_cell IS NOT NULL
      AND source_cell <> target_cell
)
SELECT
    source_cell,
    target_cell,
    COUNT(*) AS handover_count,
    MIN(event_date) AS first_seen,
    MAX(event_date) AS last_seen
FROM clean
GROUP BY 1, 2
""")

con.execute("""
CREATE OR REPLACE VIEW intercell_handover_stats AS
SELECT
    source_cell,
    target_cell,
    handover_count,
    ROUND(100.0 * handover_count / SUM(handover_count) OVER (), 4) AS pct_total_handover,
    first_seen,
    last_seen
FROM intercell_handover_counts
""")

con.execute("""
CREATE OR REPLACE VIEW daily_route_popularity AS
SELECT
    event_date,
    first_cell_id AS source_cell,
    last_cell_id AS target_cell,
    COUNT(*) AS route_count
FROM trips_with_cells
WHERE first_cell_id IS NOT NULL
  AND last_cell_id IS NOT NULL
  AND first_cell_id <> last_cell_id
GROUP BY 1, 2, 3
""")

con.execute("""
CREATE OR REPLACE VIEW daily_handover_counts AS
WITH ranked AS (
    SELECT
        vehicle_id,
        event_ts,
        event_date,
        cell_id AS target_cell,
        LAG(cell_id) OVER (PARTITION BY vehicle_id ORDER BY event_ts) AS source_cell
    FROM handover_events_typed
)
SELECT
    event_date,
    source_cell,
    target_cell,
    COUNT(*) AS handover_count
FROM ranked
WHERE source_cell IS NOT NULL
  AND target_cell IS NOT NULL
  AND source_cell <> target_cell
GROUP BY 1, 2, 3
""")

con.execute("""
SELECT 'route_popularity' AS view_name, COUNT(*) AS rows FROM route_popularity
UNION ALL
SELECT 'intercell_handover_stats', COUNT(*) FROM intercell_handover_stats
""").df()

## Section 3: Build a Natural Language Query Interface (Text -> Structured Intent)

A lightweight rules parser converts analyst prompts into a typed intent object.

In [ ]:
@dataclass
class QueryIntent:
    analysis_type: str
    metric: str
    top_k: int = 10
    start_date: str | None = None
    end_date: str | None = None
    weekday: str | None = None
    hour_start: int | None = None
    hour_end: int | None = None
    region: str | None = None
    viz: str = "table"
    sort: str = "desc"


def _extract_top_k(text: str, default: int = 10) -> int:
    m = re.search(r"\btop\s*(\d{1,3})\b", text)
    if m:
        return int(m.group(1))
    m = re.search(r"\b(\d{1,3})\s*(?:most|highest|largest)\b", text)
    if m:
        return int(m.group(1))
    return default


def _extract_last_n_days(text: str) -> tuple[str | None, str | None]:
    m = re.search(r"\blast\s*(\d{1,3})\s*days\b", text)
    if not m:
        return None, None
    n = int(m.group(1))
    end = date.today()
    start = end - timedelta(days=max(0, n - 1))
    return str(start), str(end)


def _extract_between_dates(text: str) -> tuple[str | None, str | None]:
    m = re.search(r"between\s*(\d{4}-\d{2}-\d{2})\s*and\s*(\d{4}-\d{2}-\d{2})", text)
    if not m:
        return None, None
    return m.group(1), m.group(2)


def _extract_weekday(text: str) -> str | None:
    weekdays = [
        "monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"
    ]
    for wd in weekdays:
        if wd in text:
            return wd.capitalize()
    return None


def _extract_hour_window(text: str) -> tuple[int | None, int | None]:
    m = re.search(r"(?:hour|hours)\s*(\d{1,2})\s*(?:to|-|through)\s*(\d{1,2})", text)
    if not m:
        return None, None
    h1, h2 = int(m.group(1)), int(m.group(2))
    if 0 <= h1 <= 23 and 0 <= h2 <= 23:
        return h1, h2
    return None, None


def parse_intent(prompt: str) -> QueryIntent:
    text = prompt.strip().lower()

    analysis_type = "handover_list"
    metric = "handover_count"
    viz = "table"

    if "route" in text:
        analysis_type = "route_popularity"
        metric = "route_count"
    if "map" in text:
        viz = "map"
    elif "chart" in text or "plot" in text or "bar" in text:
        viz = "bar"

    if "handover" in text or "inter-cell" in text or "intercell" in text or "transition" in text:
        analysis_type = "handover_list"
        metric = "handover_count"

    top_k = _extract_top_k(text, default=10)

    start_date, end_date = _extract_between_dates(text)
    if start_date is None:
        start_date, end_date = _extract_last_n_days(text)

    weekday = _extract_weekday(text)
    hour_start, hour_end = _extract_hour_window(text)

    region = None
    m_region = re.search(r"region\s*[:=]?\s*([a-zA-Z0-9_\-]+)", text)
    if m_region:
        region = m_region.group(1)

    return QueryIntent(
        analysis_type=analysis_type,
        metric=metric,
        top_k=max(1, min(500, top_k)),
        start_date=start_date,
        end_date=end_date,
        weekday=weekday,
        hour_start=hour_start,
        hour_end=hour_end,
        region=region,
        viz=viz,
        sort="desc",
    )


example_prompts = [
    "show me a map of the top ten most popular routes",
    "list top 20 inter-cell handovers in the last 30 days",
    "top 15 routes between 2025-01-01 and 2025-01-31",
]

parsed_examples = pd.DataFrame([asdict(parse_intent(p)) for p in example_prompts])
parsed_examples

## Section 4: Translate Intent to Safe SQL with Validation Rules

Intent fields are mapped to restricted SQL templates only. Raw SQL input is not executed.

In [ ]:
ALLOWED_ANALYSIS_TYPES = {"route_popularity", "handover_list"}
ALLOWED_VIZ = {"table", "bar", "map"}


def _validate_intent(intent: QueryIntent) -> None:
    if intent.analysis_type not in ALLOWED_ANALYSIS_TYPES:
        raise ValueError(f"Unsupported analysis_type: {intent.analysis_type}")
    if intent.viz not in ALLOWED_VIZ:
        raise ValueError(f"Unsupported viz: {intent.viz}")
    if intent.top_k < 1 or intent.top_k > 500:
        raise ValueError("top_k out of allowed range [1, 500]")


def _build_filter_parts(intent: QueryIntent, date_col: str, ts_col: str | None = None) -> list[str]:
    parts: list[str] = []

    if intent.start_date and intent.end_date:
        parts.append(
            f"{date_col} BETWEEN DATE '{intent.start_date}' AND DATE '{intent.end_date}'"
        )

    if intent.weekday and ts_col:
        parts.append(f"dayname({ts_col}) = '{intent.weekday}'")

    if intent.hour_start is not None and intent.hour_end is not None and ts_col:
        parts.append(
            f"EXTRACT(HOUR FROM {ts_col}) BETWEEN {intent.hour_start} AND {intent.hour_end}"
        )

    return parts


def _where_clause(parts: list[str]) -> str:
    if not parts:
        return ""
    return "WHERE " + " AND ".join(parts)


def build_sql(intent: QueryIntent) -> str:
    _validate_intent(intent)

    if intent.analysis_type == "route_popularity":
        filters = _build_filter_parts(intent, date_col="event_date", ts_col="trip_start")
        filters.extend([
            "first_cell_id IS NOT NULL",
            "last_cell_id IS NOT NULL",
            "first_cell_id <> last_cell_id",
        ])
        where = _where_clause(filters)

        return f"""
            SELECT
                first_cell_id AS source_cell,
                last_cell_id AS target_cell,
                COUNT(*) AS route_count,
                COUNT(DISTINCT vehicle_id) AS unique_vehicles,
                AVG(duration_minutes) AS avg_duration_min,
                AVG(n_handovers) AS avg_handovers,
                MIN(event_date) AS first_seen,
                MAX(event_date) AS last_seen
            FROM trips_with_cells
            {where}
            GROUP BY 1, 2
            ORDER BY route_count DESC
            LIMIT {intent.top_k}
        """

    filters = _build_filter_parts(intent, date_col="event_date", ts_col="event_ts")
    filters.extend([
        "source_cell IS NOT NULL",
        "target_cell IS NOT NULL",
        "source_cell <> target_cell",
    ])
    where = _where_clause(filters)

    return f"""
        WITH ranked AS (
            SELECT
                vehicle_id,
                event_ts,
                event_date,
                cell_id AS target_cell,
                LAG(cell_id) OVER (PARTITION BY vehicle_id ORDER BY event_ts) AS source_cell
            FROM handover_events_typed
        )
        SELECT
            source_cell,
            target_cell,
            COUNT(*) AS handover_count,
            ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct_total_handover,
            MIN(event_date) AS first_seen,
            MAX(event_date) AS last_seen
        FROM ranked
        {where}
        GROUP BY 1, 2
        ORDER BY handover_count DESC
        LIMIT {intent.top_k}
    """

## Section 5: Execute Queries and Return Typed Results

Execution includes structured error handling for malformed intents and empty result sets.

In [ ]:
def run_query(prompt: str, connection: duckdb.DuckDBPyConnection = con) -> tuple[QueryIntent, str, pd.DataFrame]:
    try:
        intent = parse_intent(prompt)
        sql = build_sql(intent)
        result = connection.execute(sql).df()
        return intent, sql, result
    except Exception as exc:
        err_df = pd.DataFrame({"error": [str(exc)], "prompt": [prompt]})
        fallback_intent = QueryIntent(
            analysis_type="handover_list",
            metric="handover_count",
            top_k=10,
            viz="table",
        )
        return fallback_intent, "", err_df


def show_result_table(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame({"message": ["No rows returned for this query."]})
    return df

## Section 6: Visualize Top 10 Most Popular Routes on an Interactive Map

If a coordinates table with `cell_id`, `lat`, and `lon` is found in the analytics DB, a geo map is rendered. Otherwise, a ranked bar chart is shown as fallback.

In [ ]:
def _has_cell_locations(connection: duckdb.DuckDBPyConnection = con) -> bool:
    """Return True if a cell_locations view/table with lat/lon is available."""
    try:
        connection.execute("SELECT cell_id, lat, lon FROM cell_locations LIMIT 1")
        return True
    except Exception:
        return False


def _plot_route_map_or_fallback(
    routes_df: pd.DataFrame,
    connection: duckdb.DuckDBPyConnection = con,
) -> go.Figure:
    if routes_df.empty:
        return px.bar(title="No routes found for this query")

    if _has_cell_locations(connection):
        coords = connection.execute("SELECT cell_id, lat, lon FROM cell_locations").df()

        merged = routes_df.merge(
            coords.rename(columns={"cell_id": "src_cell_id", "lat": "src_lat", "lon": "src_lon"}),
            left_on="source_cell", right_on="src_cell_id", how="left",
        )
        merged = merged.merge(
            coords.rename(columns={"cell_id": "dst_cell_id", "lat": "dst_lat", "lon": "dst_lon"}),
            left_on="target_cell", right_on="dst_cell_id", how="left",
        )
        merged_geo = merged.dropna(subset=["src_lat", "src_lon", "dst_lat", "dst_lon"]).reset_index(drop=True)

        if merged_geo.empty:
            print("⚠ Routes found but none matched cell_locations coordinates — showing bar fallback.")
        else:
            max_count = float(merged_geo["route_count"].max()) or 1.0
            fig = go.Figure()

            for rank, row in merged_geo.iterrows():
                weight = max(1.5, float(row["route_count"]) / max_count * 10)
                fig.add_trace(
                    go.Scattermap(
                        mode="lines+markers",
                        lon=[row["src_lon"], row["dst_lon"]],
                        lat=[row["src_lat"], row["dst_lat"]],
                        line={"width": weight},
                        marker={"size": 6},
                        name=f"#{rank + 1}: {int(row['source_cell'])}→{int(row['target_cell'])}",
                        hovertemplate=(
                            f"<b>Rank #{rank + 1}</b><br>"
                            f"Route: {int(row['source_cell'])} → {int(row['target_cell'])}<br>"
                            f"Trips: {int(row['route_count'])}<br>"
                            f"Unique vehicles: {int(row['unique_vehicles'])}<br>"
                            f"Avg duration: {row['avg_duration_min']:.1f} min<extra></extra>"
                        ),
                    )
                )

            # compute a reasonable map centre
            mid_lat = (merged_geo["src_lat"].mean() + merged_geo["dst_lat"].mean()) / 2
            mid_lon = (merged_geo["src_lon"].mean() + merged_geo["dst_lon"].mean()) / 2

            fig.update_layout(
                title=f"Top {len(merged_geo)} Most Popular Routes",
                map={
                    "style": "carto-positron",
                    "center": {"lat": mid_lat, "lon": mid_lon},
                    "zoom": 9,
                },
                margin={"l": 0, "r": 0, "t": 40, "b": 0},
                legend={"title": "Route (ranked)"},
            )
            return fig

    # ── Fallback: ranked horizontal bar chart ────────────────────────────────
    ranked = routes_df.copy()
    ranked["rank"] = range(1, len(ranked) + 1)
    ranked["route"] = "#" + ranked["rank"].astype(str) + ": " + ranked["source_cell"].astype(str) + " → " + ranked["target_cell"].astype(str)
    fig = px.bar(
        ranked,
        x="route_count",
        y="route",
        orientation="h",
        title="Top Routes by Frequency (no coordinates available)",
        labels={"route_count": "Trips", "route": "Route"},
        color="route_count",
        color_continuous_scale="Blues",
    )
    fig.update_layout(yaxis={"categoryorder": "total ascending"}, coloraxis_showscale=False)
    return fig


intent_map, sql_map, df_map = run_query("show me a map of the top 10 most popular routes")
fig_map = _plot_route_map_or_fallback(df_map)
fig_map.show()
show_result_table(df_map.head(10))

## Section 7: Generate Ranked List of Most Common Inter-Cell Handovers

Transition-frequency output includes count and percent share.

In [ ]:
intent_ho, sql_ho, df_ho = run_query("generate a list of the top 25 most common inter-cell handovers")
print(sql_ho)
show_result_table(df_ho.head(25))

## Section 8: Add Reusable Prompt Templates for Common Analyst Questions

In [ ]:
PROMPT_TEMPLATES = {
    "top_routes_by_weekday": "show top {k} routes on {weekday}",
    "handover_hotspots_region": "list top {k} inter-cell handovers in region {region}",
    "compare_routes_weeks": "top {k} routes in last 7 days",
}


def render_prompt(template_name: str, **kwargs) -> str:
    if template_name not in PROMPT_TEMPLATES:
        raise KeyError(f"Unknown template: {template_name}")
    return PROMPT_TEMPLATES[template_name].format(**kwargs)


example_template_prompts = [
    render_prompt("top_routes_by_weekday", k=10, weekday="Monday"),
    render_prompt("handover_hotspots_region", k=15, region="north"),
    render_prompt("compare_routes_weeks", k=20),
]

pd.DataFrame({"template_prompt": example_template_prompts})

## Section 9: Evaluate Query Accuracy with Notebook Test Cases

In [ ]:
TEST_CASES = [
    {
        "prompt": "show me a map of the top ten most popular routes",
        "expected_analysis_type": "route_popularity",
    },
    {
        "prompt": "top 20 inter-cell handovers",
        "expected_analysis_type": "handover_list",
    },
    {
        "prompt": "top 12 routes between 2025-01-01 and 2025-01-31",
        "expected_analysis_type": "route_popularity",
    },
]

results = []
for tc in TEST_CASES:
    intent = parse_intent(tc["prompt"])
    ok_type = intent.analysis_type == tc["expected_analysis_type"]

    sql = build_sql(intent)
    no_unsafe = all(x not in sql.lower() for x in [";", " drop ", " delete ", " update ", " insert "])

    try:
        df = con.execute(sql).df()
        ran = True
        row_count = len(df)
    except Exception:
        ran = False
        row_count = -1

    results.append(
        {
            "prompt": tc["prompt"],
            "intent_type_ok": ok_type,
            "sql_safe_ok": no_unsafe,
            "query_ran_ok": ran,
            "rows_returned": row_count,
        }
    )

test_df = pd.DataFrame(results)
test_df

## Section 10: Package the Interface as Reusable Notebook Functions

`NLMobilityQueryEngine` wraps parse/build/run/render into a reusable object.

In [ ]:
class NLMobilityQueryEngine:
    def __init__(self, connection: duckdb.DuckDBPyConnection):
        self.con = connection

    def parse_intent(self, prompt: str) -> QueryIntent:
        return parse_intent(prompt)

    def build_sql(self, intent: QueryIntent) -> str:
        return build_sql(intent)

    def run_query(self, prompt: str) -> tuple[QueryIntent, str, pd.DataFrame]:
        return run_query(prompt, connection=self.con)

    def render_map(self, routes_df: pd.DataFrame) -> go.Figure:
        return _plot_route_map_or_fallback(routes_df, connection=self.con)

    def render_table(self, df: pd.DataFrame) -> pd.DataFrame:
        return show_result_table(df)

    def has_coordinates(self) -> bool:
        return _has_cell_locations(self.con)


engine = NLMobilityQueryEngine(con)
print(f"Coordinates available: {engine.has_coordinates()}")

prompt = "show me a map of the top 10 most popular routes in last 14 days"
intent, sql, df = engine.run_query(prompt)
print("Intent:", asdict(intent))
print("SQL:\n", sql)

if intent.analysis_type == "route_popularity" and intent.viz == "map":
    engine.render_map(df).show()

engine.render_table(df.head(intent.top_k))